Load all tables

In [1]:
import pandas as pd
import numpy as np

customers = pd.read_csv("Data/customers.csv")
orders = pd.read_csv("Data/orders.csv")
order_items = pd.read_csv("Data/order_items.csv")
products = pd.read_csv("Data/products.csv")
campaigns = pd.read_csv("Data/marketing_campaigns.csv")
website_events = pd.read_csv("Data/website_events.csv")
support = pd.read_csv("Data/customer_support.csv")

Check whether everything loaded

In [2]:
tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "campaigns": campaigns,
    "website_events": website_events,
    "support": support
}

for name, df in tables.items():
    print(f"{name:20} {df.shape}")

customers            (50000, 9)
orders               (220000, 9)
order_items          (424819, 6)
products             (1000, 7)
campaigns            (24, 7)
website_events       (650000, 7)
support              (45000, 7)


Understand the columns

In [3]:
for name, df in tables.items():
    print("\n" + "=" * 60)
    print(name.upper())
    print("=" * 60)
    print(df.info())


CUSTOMERS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   customer_id          50000 non-null  object
 1   signup_date          50000 non-null  object
 2   gender               50000 non-null  object
 3   age                  50000 non-null  int64 
 4   city                 50000 non-null  object
 5   state                50000 non-null  object
 6   acquisition_channel  50000 non-null  object
 7   customer_type        50000 non-null  object
 8   income_band          50000 non-null  object
dtypes: int64(1), object(8)
memory usage: 3.4+ MB
None

ORDERS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220000 entries, 0 to 219999
Data columns (total 9 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         220000 non-null  object 
 1   customer_id      220000

Convert date columns

In [4]:
customers["signup_date"] = pd.to_datetime(customers["signup_date"])

orders["order_date"] = pd.to_datetime(orders["order_date"])

campaigns["start_date"] = pd.to_datetime(campaigns["start_date"])
campaigns["end_date"] = pd.to_datetime(campaigns["end_date"])

website_events["event_time"] = pd.to_datetime(website_events["event_time"])

support["ticket_date"] = pd.to_datetime(support["ticket_date"])

In [5]:
print(customers.dtypes)
print(orders.dtypes)

customer_id                    object
signup_date            datetime64[ns]
gender                         object
age                             int64
city                           object
state                          object
acquisition_channel            object
customer_type                  object
income_band                    object
dtype: object
order_id                   object
customer_id                object
order_date         datetime64[ns]
payment_method             object
shipping_type              object
order_status               object
total_amount              float64
discount_amount           float64
shipping_cost             float64
dtype: object


Check missing values

In [6]:
for name, df in tables.items():
    print("\n", name)
    print(df.isnull().sum())


 customers
customer_id            0
signup_date            0
gender                 0
age                    0
city                   0
state                  0
acquisition_channel    0
customer_type          0
income_band            0
dtype: int64

 orders
order_id           0
customer_id        0
order_date         0
payment_method     0
shipping_type      0
order_status       0
total_amount       0
discount_amount    0
shipping_cost      0
dtype: int64

 order_items
order_id         0
product_id       0
quantity         0
selling_price    0
cost_price       0
discount         0
dtype: int64

 products
product_id      0
product_name    0
category        0
sub_category    0
brand           0
cost_price      0
list_price      0
dtype: int64

 campaigns
campaign_id       0
campaign_name     0
channel           0
start_date        0
end_date          0
campaign_cost     0
target_segment    0
dtype: int64

 website_events
event_id       0
customer_id    0
event_time     0
event_type     

Check duplicate records

In [7]:
for name, df in tables.items():
    print(
        f"{name:20} "
        f"Rows = {len(df):8,} | "
        f"Duplicates = {df.duplicated().sum():6,}"
    )

customers            Rows =   50,000 | Duplicates =      0
orders               Rows =  220,000 | Duplicates =      0
order_items          Rows =  424,819 | Duplicates =      0
products             Rows =    1,000 | Duplicates =      0
campaigns            Rows =       24 | Duplicates =      0
website_events       Rows =  650,000 | Duplicates =      0
support              Rows =   45,000 | Duplicates =      0


Check primary keys

In [8]:
# Check uniqueness of primary keys

primary_keys = {
    "customers": "customer_id",
    "orders": "order_id",
    "products": "product_id",
    "campaigns": "campaign_id",
    "website_events": "event_id",
    "support": "ticket_id"
}

for table_name, key in primary_keys.items():
    df = tables[table_name]
    
    print(
        f"{table_name:20} "
        f"Total Rows = {len(df):8,} | "
        f"Unique {key} = {df[key].nunique():8,}"
    )

customers            Total Rows =   50,000 | Unique customer_id =   50,000
orders               Total Rows =  220,000 | Unique order_id =  220,000
products             Total Rows =    1,000 | Unique product_id =    1,000
campaigns            Total Rows =       24 | Unique campaign_id =       24
website_events       Total Rows =  650,000 | Unique event_id =  650,000
support              Total Rows =   45,000 | Unique ticket_id =   45,000


Check relationships between tables

In [9]:
# Check whether every order belongs to a valid customer

invalid_customers = ~orders["customer_id"].isin(customers["customer_id"])

print("Invalid customer IDs in orders:", invalid_customers.sum())

Invalid customer IDs in orders: 0


In [10]:
# Check whether every order item belongs to a valid order

invalid_orders = ~order_items["order_id"].isin(orders["order_id"])

print("Invalid order IDs in order_items:", invalid_orders.sum())

Invalid order IDs in order_items: 0


In [11]:
# Check whether every order item belongs to a valid product

invalid_products = ~order_items["product_id"].isin(products["product_id"])

print("Invalid product IDs in order_items:", invalid_products.sum())

Invalid product IDs in order_items: 0


In [12]:
# Check whether every website event belongs to a valid customer

invalid_event_customers = ~website_events["customer_id"].isin(customers["customer_id"])

print("Invalid customer IDs in website_events:",
      invalid_event_customers.sum())

Invalid customer IDs in website_events: 0


In [13]:
# Check whether every support ticket belongs to a valid customer

invalid_support_customers = ~support["customer_id"].isin(customers["customer_id"])

print("Invalid customer IDs in support:",
      invalid_support_customers.sum())

Invalid customer IDs in support: 0


Create our first business summary

In [14]:
# Basic business overview

summary = {
    "Total Customers": customers["customer_id"].nunique(),
    "Total Orders": orders["order_id"].nunique(),
    "Total Products": products["product_id"].nunique(),
    "Total Website Events": website_events["event_id"].nunique(),
    "Total Support Tickets": support["ticket_id"].nunique(),
    "Total Revenue": orders["total_amount"].sum(),
    "Total Discounts": orders["discount_amount"].sum(),
    "Completed Orders": (orders["order_status"] == "Completed").sum(),
    "Cancelled Orders": (orders["order_status"] == "Cancelled").sum(),
    "Returned Orders": (orders["order_status"] == "Returned").sum()
}

for metric, value in summary.items():
    if isinstance(value, (int, np.integer)):
        print(f"{metric:25}: {value:,}")
    else:
        print(f"{metric:25}: {value:,.2f}")

Total Customers          : 50,000
Total Orders             : 220,000
Total Products           : 1,000
Total Website Events     : 650,000
Total Support Tickets    : 45,000
Total Revenue            : 1,175,489,360.90
Total Discounts          : 131,699,395.02
Completed Orders         : 195,930
Cancelled Orders         : 13,182
Returned Orders          : 10,888
